In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import glob
from time import time

from RaTag.core.datatypes import Run, SetPmt

from RaTag.io.bootstrap import bootstrap_from_config
from RaTag.el_tpc.drift_workflow import map_drift_physics
from RaTag.el_tpc.timing_workflow import map_time_windows, resolve_set_timing, map_timing_plots

%matplotlib inline

# Initialize test run and set

In [3]:
# raw_data_path = '/Users/pabloherrero/sabat/RaTagging/test_data/raw_data/RUN8/'
# raw_data_path = "/Volumes/KINGSTON/RaTag_data/raw_waveforms/RUN8_EL2350Vcm_5GSsec"
# config_path = '/Users/pabloherrero/sabat/RaTagging/test_data/configs/run8_analysis.yaml'
# config_path = '/Users/pabloherrero/sabat/RaTagging/configs/recoils/run8_analysis.yaml'
config_path = '/Users/pabloherrero/sabat/RaTagging/configs/recoils/run26_analysis.yaml'
bare_run = bootstrap_from_config(config_path)

Found 8 set directories in RUN26_Th228_EL2700: ['FieldScan_Gate0050_Anode2750', 'FieldScan_Gate0100_Anode2800', 'FieldScan_Gate0200_Anode2900', 'FieldScan_Gate0300_Anode3000', 'FieldScan_Gate0500_Anode3200', 'FieldScan_Gate0700_Anode3400', 'FieldScan_Gate1000_Anode3700', 'FieldScan_Gate1200_Anode3900']
  Found 400 .wfm files in FieldScan_Gate0050_Anode2750
  Found 400 .wfm files in FieldScan_Gate0100_Anode2800
  Found 400 .wfm files in FieldScan_Gate0200_Anode2900
  Found 400 .wfm files in FieldScan_Gate0300_Anode3000
  Found 400 .wfm files in FieldScan_Gate0500_Anode3200
  Found 400 .wfm files in FieldScan_Gate0700_Anode3400
  Found 400 .wfm files in FieldScan_Gate1000_Anode3700
  Found 400 .wfm files in FieldScan_Gate1200_Anode3900


In [4]:
set0 = bare_run.sets[0]
print(set0)

SetPmt state:
  source_dir = /Volumes/KINGSTON/RaTag_data/raw_waveforms/RUN26_Th228_EL2700/FieldScan_Gate0050_Anode2750
  filenames = <400 files, 19200 waveforms, FastFrame(48 frames/file)>
  multiiso = False
  ff = True
  nframes = 48
  gate = 50
  anode = 2750

  Missing:
    sampling_rate, drift_field, EL_field, red_drift_field, red_EL_field, speed_drift, time_drift, diffusion_coefficient, baseline_median, baseline_std, t_s1, t_s1_std, t_s2_start, t_s2_start_std, t_s2_end, t_s2_end_std, n_areas_recoil, area_s2_mean, area_s2_sigma, area_s2_ci95, area_s2_fit_success


# Set drift properties

In [5]:
run = map_drift_physics(bare_run, force=True)
set0 = run.sets[0]

  ⚡ Force enabled: Recomputing drift_field
  ✓ FieldScan_Gate0050_Anode2750: Computed 'drift_field' and cached
  ⚡ Force enabled: Recomputing red_EL_field
  ✓ FieldScan_Gate0050_Anode2750: Computed 'red_EL_field' and cached
  ⚡ Force enabled: Recomputing time_drift
  ✓ FieldScan_Gate0050_Anode2750: Computed 'time_drift' and cached
  ⚡ Force enabled: Recomputing drift_field
  ✓ FieldScan_Gate0100_Anode2800: Computed 'drift_field' and cached
  ⚡ Force enabled: Recomputing red_EL_field
  ✓ FieldScan_Gate0100_Anode2800: Computed 'red_EL_field' and cached
  ⚡ Force enabled: Recomputing time_drift
  ✓ FieldScan_Gate0100_Anode2800: Computed 'time_drift' and cached
  ⚡ Force enabled: Recomputing drift_field
  ✓ FieldScan_Gate0200_Anode2900: Computed 'drift_field' and cached
  ⚡ Force enabled: Recomputing red_EL_field
  ✓ FieldScan_Gate0200_Anode2900: Computed 'red_EL_field' and cached
  ⚡ Force enabled: Recomputing time_drift
  ✓ FieldScan_Gate0200_Anode2900: Computed 'time_drift' and cached
 

In [17]:
print(set0)

SetPmt state:
  source_dir = /Volumes/KINGSTON/RaTag_data/raw_waveforms/RUN26_Th228_EL2700/FieldScan_Gate0050_Anode2750
  filenames = <400 files, 19200 waveforms, FastFrame(48 frames/file)>
  multiiso = False
  ff = True
  nframes = 48
  gate = 50
  anode = 2750
  drift_field = 35.714285714285715
  EL_field = 3375.0
  red_drift_field = 0.07322370589285715
  red_EL_field = 6.919640206875001
  speed_drift = 0.7008551951440259
  time_drift = 19.975595667979597

  Missing:
    sampling_rate, diffusion_coefficient, baseline_median, baseline_std, t_s1, t_s1_std, t_s2_start, t_s2_start_std, t_s2_end, t_s2_end_std, n_areas_recoil, area_s2_mean, area_s2_sigma, area_s2_ci95, area_s2_fit_success


# Test new timing_workflow

## Individual sets

In [19]:
set8 = run.sets[7]
set8t = resolve_set_timing(set8, max_frames=500, threshold_s1=0.5, threshold_s2=0.8, window_size=9, threshold_bs=0.02, force=True)

  ⚡ Force enabled: Recomputing t_s2_end
    💾 Saved to RUN26_Th228_EL2700/FieldScan_Gate1200_Anode3900_timing.npz
  ✓ FieldScan_Gate1200_Anode3900: Computed 't_s2_end' and cached


## Full run

In [ ]:
runt = map_time_windows(run, max_frames=48*400, force=True)

  ⚡ Force enabled: Recomputing t_s2_end
    💾 Saved to RUN26_Th228_EL2700/FieldScan_Gate0050_Anode2750_timing.npz
  ✓ FieldScan_Gate0050_Anode2750: Computed 't_s2_end' and cached
  ⚡ Force enabled: Recomputing t_s2_end
    💾 Saved to RUN26_Th228_EL2700/FieldScan_Gate0100_Anode2800_timing.npz
  ✓ FieldScan_Gate0100_Anode2800: Computed 't_s2_end' and cached
  ⚡ Force enabled: Recomputing t_s2_end
    💾 Saved to RUN26_Th228_EL2700/FieldScan_Gate0200_Anode2900_timing.npz
  ✓ FieldScan_Gate0200_Anode2900: Computed 't_s2_end' and cached
  ⚡ Force enabled: Recomputing t_s2_end
    💾 Saved to RUN26_Th228_EL2700/FieldScan_Gate0300_Anode3000_timing.npz
  ✓ FieldScan_Gate0300_Anode3000: Computed 't_s2_end' and cached
  ⚡ Force enabled: Recomputing t_s2_end
    💾 Saved to RUN26_Th228_EL2700/FieldScan_Gate0500_Anode3200_timing.npz
  ✓ FieldScan_Gate0500_Anode3200: Computed 't_s2_end' and cached
  ⚡ Force enabled: Recomputing t_s2_end
    💾 Saved to RUN26_Th228_EL2700/FieldScan_Gate0700_Anode3400_ti

In [21]:
runt = map_timing_plots(runt, force=True)

  ⚡ Force enabled: Regenerating plots for RUN_26

GENERATING TIMING PLOTS: RUN_26
  → Saved: /Volumes/KINGSTON/RaTag_data/processed/RUN26_Th228_EL2700/plots/timing_qa/RUN_26_histograms.png
  → Saved: /Volumes/KINGSTON/RaTag_data/processed/RUN26_Th228_EL2700/plots/timing_qa/RUN_26_timing_vs_field.png
  → Saved: /Volumes/KINGSTON/RaTag_data/processed/RUN26_Th228_EL2700/plots/timing_qa/RUN_26_validation.png


# Integration pipeline A/B test

In [ ]:
from RaTag.io.file_ops import iter_waveforms
from RaTag.core.paths import get_output_root
from RaTag.waveform.preprocessing import subtract_pedestal, moving_average, threshold_clip

def _integrate_s2_in_set_AB_TEST(set_pmt: SetPmt,
                                 max_files: int,
                                 integration_config: IntegrationConfig,
                                 use_dynamic_window: bool = False) -> S2Areas:
    """
    Vectorized A/B test for S2 Integration.
    """
    # 1. Setup Data Buffers
    out_uids, out_areas = [], []
    
    # 2. Setup Static Bounds (Option A)
    s2_start_static = float(set_pmt.t_s2_start) - (integration_config.n_sigma_start * float(set_pmt.t_s2_start_std or 0))
    s2_end_static = float(set_pmt.t_s2_end) + (integration_config.n_sigma_end * float(set_pmt.t_s2_end_std or 0))
    
    # 3. Load Dynamic Payload (Option B)
    if use_dynamic_window:
        data_file = get_output_root(set_pmt.source_dir.parent) / f"{set_pmt.source_dir.name}_timing.npz"
        payload = dict(np.load(data_file))
        # Create fast lookup dictionaries for per-frame timing
        dict_starts = dict(zip(payload['uids'], payload['t_s2_start']))
        dict_ends = dict(zip(payload['uids'], payload['t_s2_end']))

    # 4. THE 2D VECTORIZED LOOP
    for wf in iter_waveforms(set_pmt, max_files=max_files):
        # Native 2D Preprocessing
        wf = subtract_pedestal(wf, n_points=integration_config.n_pedestal)
        wf = moving_average(wf, window=integration_config.ma_window)
        wf = threshold_clip(wf, threshold=integration_config.bs_threshold)

        if use_dynamic_window:
            # --- OPTION B: Dynamic Per-Frame Masking ---
            # Lookup the specific bounds for these 48 frames
            chunk_starts = np.array([dict_starts.get(u, np.nan) for u in wf.uids])
            chunk_ends = np.array([dict_ends.get(u, np.nan) for u in wf.uids])
            
            # Create a 2D Boolean Mask using NumPy Broadcasting
            time_2d = wf.t[None, :] # Shape: (1, n_samples)
            mask_2d = (time_2d >= chunk_starts[:, None]) & (time_2d <= chunk_ends[:, None])
            
            # Apply mask: Keep voltage where True, set to 0.0 where False
            v_ready = np.where(mask_2d, wf.v, 0.0)
            
            # Mark frames that had no valid timing data so we don't integrate them
            valid_frames = ~np.isnan(chunk_starts)
            
        else:
            # --- OPTION A: Static Global Masking ---
            mask_1d = (wf.t >= s2_start_static) & (wf.t <= s2_end_static)
            v_ready = wf.v[:, mask_1d] if wf.ff else wf.v[mask_1d][np.newaxis, :]
            valid_frames = np.ones(wf.nframes, dtype=bool) # All frames are valid in static

        # 5. Native 2D Trapezoidal Integration (Hits all frames at once)
        chunk_areas = np.trapezoid(v_ready, dx=integration_config.dt, axis=1)

        # 6. Collect Valid Results
        out_uids.append(wf.uids[valid_frames])
        out_areas.append(chunk_areas[valid_frames])

    # 7. Flatten and Return
    areas = np.concatenate(out_areas)
    uids = np.concatenate(out_uids)

    return S2Areas(
        source_dir=set_pmt.source_dir,
        areas=areas,
        uids=uids,
        method=f"recoil_integration_{'dynamic' if use_dynamic_window else 'static'}",
        params={}
    )

In [20]:
from dataclasses import replace

from RaTag.workflows.recoil_integration import _fit_and_save_s2_histogram, _integrate_s2_in_set_AB_TEST
from RaTag.core.config import IntegrationConfig, FitConfig
from RaTag.core.dataIO import store_s2area

from RaTag.core.paths import get_output_root

integ_dict = {
    'n_sigma_start': 2,
    'n_sigma_end': 2,
    'n_pedestal': 100,
    'ma_window': 9,
    'bs_threshold': 0.03,
    'dt': 2e-4
}

integ_config = IntegrationConfig(**integ_dict)

fit_config = FitConfig(
    bin_cuts=[0, 10],
    nbins=100,
    exclude_index=1,
    bg_threshold=0.5,
    bg_cutoff=0.07,
    n_sigma=3.,
    upper_limit=5.0)

## Static window (classic)

In [22]:
updated_sets = []
for set_pmt in runt.sets:
    print(f"Processing set {set_pmt.source_dir.name}...")
    s2_area = _integrate_s2_in_set_AB_TEST(set_pmt, max_files=48*400,
                                            integration_config=integ_config,
                                            use_dynamic_window=False)
    # store_s2area(set_pmt, s2_area)

    plots_dir = get_output_root(run.root_directory) / "plots" / "s2_areas"
    plots_dir.mkdir(parents=True, exist_ok=True)

    updated_set = _fit_and_save_s2_histogram(set_pmt, s2_area, 
                                             fit_config=fit_config,
                                            plots_dir=plots_dir)
    updated_sets.append(updated_set)


    run_static = replace(runt, sets=updated_sets)

Processing set FieldScan_Gate0050_Anode2750...
  Background detection: 37.6% of counts in low region
  Recommendation: two_stage
  Using two-stage fitting with background subtraction...
    Fitting background Gaussian in range [0, 0.07] mV·µs
User defined signal region: [0.05, 5.0]
    ✓ Fit (two_stage): μ=0.050 ± 1.274 mV·µs
    💾 Saved explicit fit result to FieldScan_Gate0050_Anode2750_s2_areas_hist_fit.json
  → Saved: /Volumes/KINGSTON/RaTag_data/processed/RUN26_Th228_EL2700/plots/s2_areas/FieldScan_Gate0050_Anode2750_s2_histogram.png
    📊 Saved histogram plot
Processing set FieldScan_Gate0100_Anode2800...
  Background detection: 25.7% of counts in low region
  Recommendation: two_stage
  Using two-stage fitting with background subtraction...
    Fitting background Gaussian in range [0, 0.07] mV·µs
User defined signal region: [0.05000000000000003, 5.0]
    ✓ Fit (two_stage): μ=0.372 ± 0.000 mV·µs
    💾 Saved explicit fit result to FieldScan_Gate0100_Anode2800_s2_areas_hist_fit.jso

/Users/pabloherrero/sabat/RaTagging/RaTag/core/fitting.py:303: RuntimeWarning: invalid value encountered in power
  tail = A_tail / (denom_safe)**m


  → Saved: /Volumes/KINGSTON/RaTag_data/processed/RUN26_Th228_EL2700/plots/s2_areas/FieldScan_Gate0100_Anode2800_s2_histogram.png
    📊 Saved histogram plot
Processing set FieldScan_Gate0200_Anode2900...
  Background detection: 19.9% of counts in low region
  Recommendation: two_stage
  Using two-stage fitting with background subtraction...
    Fitting background Gaussian in range [0, 0.07] mV·µs
User defined signal region: [0.05, 5.0]
    ✓ Fit (two_stage): μ=1.136 ± 0.000 mV·µs
    💾 Saved explicit fit result to FieldScan_Gate0200_Anode2900_s2_areas_hist_fit.json
  → Saved: /Volumes/KINGSTON/RaTag_data/processed/RUN26_Th228_EL2700/plots/s2_areas/FieldScan_Gate0200_Anode2900_s2_histogram.png
    📊 Saved histogram plot
Processing set FieldScan_Gate0300_Anode3000...
  Background detection: 17.1% of counts in low region
  Recommendation: two_stage
  Using two-stage fitting with background subtraction...
    Fitting background Gaussian in range [0, 0.07] mV·µs
User defined signal region: 

/Users/pabloherrero/sabat/RaTagging/RaTag/core/fitting.py:303: RuntimeWarning: invalid value encountered in power
  tail = A_tail / (denom_safe)**m


  → Saved: /Volumes/KINGSTON/RaTag_data/processed/RUN26_Th228_EL2700/plots/s2_areas/FieldScan_Gate0300_Anode3000_s2_histogram.png
    📊 Saved histogram plot
Processing set FieldScan_Gate0500_Anode3200...
  Background detection: 15.2% of counts in low region
  Recommendation: two_stage
  Using two-stage fitting with background subtraction...
    Fitting background Gaussian in range [0, 0.07] mV·µs
User defined signal region: [0.05, 5.0]
    ✓ Fit (two_stage): μ=2.344 ± 0.000 mV·µs
    💾 Saved explicit fit result to FieldScan_Gate0500_Anode3200_s2_areas_hist_fit.json
  → Saved: /Volumes/KINGSTON/RaTag_data/processed/RUN26_Th228_EL2700/plots/s2_areas/FieldScan_Gate0500_Anode3200_s2_histogram.png
    📊 Saved histogram plot
Processing set FieldScan_Gate0700_Anode3400...
  Background detection: 14.1% of counts in low region
  Recommendation: two_stage
  Using two-stage fitting with background subtraction...
    Fitting background Gaussian in range [0, 0.07] mV·µs
User defined signal region: 

## Dynamic per-frame window (original)

In [21]:
updated_sets_dyn = []
for set_pmt in runt.sets:
    print(f"Processing set {set_pmt.source_dir.name}...")
    s2_area = _integrate_s2_in_set_AB_TEST(set_pmt, max_files=48*400,
                                            integration_config=integ_config,
                                            use_dynamic_window=True)
    # store_s2area(set_pmt, s2_area)

    plots_dir = get_output_root(run.root_directory) / "plots" / "s2_areas"
    plots_dir.mkdir(parents=True, exist_ok=True)

    updated_set = _fit_and_save_s2_histogram(set_pmt, s2_area, 
                                             fit_config=fit_config, 
                                             plots_dir=plots_dir)
    updated_sets_dyn.append(updated_set)


    run_dyn = replace(runt, sets=updated_sets_dyn)

Processing set FieldScan_Gate0050_Anode2750...
  Background detection: 49.0% of counts in low region
  Recommendation: two_stage
  Using two-stage fitting with background subtraction...
    Fitting background Gaussian in range [0, 0.07] mV·µs
User defined signal region: [0.05, 5.0]
    ✓ Fit (two_stage): μ=0.050 ± 1.113 mV·µs
    💾 Saved explicit fit result to FieldScan_Gate0050_Anode2750_s2_areas_hist_fit.json
  → Saved: /Volumes/KINGSTON/RaTag_data/processed/RUN26_Th228_EL2700/plots/s2_areas/FieldScan_Gate0050_Anode2750_s2_histogram.png
    📊 Saved histogram plot
Processing set FieldScan_Gate0100_Anode2800...
  Background detection: 33.9% of counts in low region
  Recommendation: two_stage
  Using two-stage fitting with background subtraction...
    Fitting background Gaussian in range [0, 0.07] mV·µs
User defined signal region: [0.05, 5.0]
    ✓ Fit (two_stage): μ=0.050 ± 0.076 mV·µs
    💾 Saved explicit fit result to FieldScan_Gate0100_Anode2800_s2_areas_hist_fit.json


/Users/pabloherrero/sabat/RaTagging/RaTag/core/fitting.py:303: RuntimeWarning: invalid value encountered in power
  tail = A_tail / (denom_safe)**m


  → Saved: /Volumes/KINGSTON/RaTag_data/processed/RUN26_Th228_EL2700/plots/s2_areas/FieldScan_Gate0100_Anode2800_s2_histogram.png
    📊 Saved histogram plot
Processing set FieldScan_Gate0200_Anode2900...
  Background detection: 27.3% of counts in low region
  Recommendation: two_stage
  Using two-stage fitting with background subtraction...
    Fitting background Gaussian in range [0, 0.07] mV·µs
User defined signal region: [0.05, 5.0]
    ✓ Fit (two_stage): μ=0.106 ± 0.000 mV·µs
    💾 Saved explicit fit result to FieldScan_Gate0200_Anode2900_s2_areas_hist_fit.json


/Users/pabloherrero/sabat/RaTagging/RaTag/core/fitting.py:303: RuntimeWarning: invalid value encountered in power
  tail = A_tail / (denom_safe)**m


  → Saved: /Volumes/KINGSTON/RaTag_data/processed/RUN26_Th228_EL2700/plots/s2_areas/FieldScan_Gate0200_Anode2900_s2_histogram.png
    📊 Saved histogram plot
Processing set FieldScan_Gate0300_Anode3000...
  Background detection: 24.3% of counts in low region
  Recommendation: two_stage
  Using two-stage fitting with background subtraction...
    Fitting background Gaussian in range [0, 0.07] mV·µs
User defined signal region: [0.05, 5.0]
    ✓ Fit (two_stage): μ=1.310 ± 0.000 mV·µs
    💾 Saved explicit fit result to FieldScan_Gate0300_Anode3000_s2_areas_hist_fit.json


...

  → Saved: /Volumes/KINGSTON/RaTag_data/processed/RUN26_Th228_EL2700/plots/s2_areas/FieldScan_Gate0300_Anode3000_s2_histogram.png
    📊 Saved histogram plot
Processing set FieldScan_Gate0500_Anode3200...
  Background detection: 17.6% of counts in low region
  Recommendation: two_stage
  Using two-stage fitting with background subtraction...
    Fitting background Gaussian in range [0, 0.07] mV·µs
User defined signal region: [0.05, 5.0]
    ✓ Fit (two_stage): μ=2.056 ± 0.000 mV·µs
    💾 Saved explicit fit result to FieldScan_Gate0500_Anode3200_s2_areas_hist_fit.json
  → Saved: /Volumes/KINGSTON/RaTag_data/processed/RUN26_Th228_EL2700/plots/s2_areas/FieldScan_Gate0500_Anode3200_s2_histogram.png
    📊 Saved histogram plot
Processing set FieldScan_Gate0700_Anode3400...
  Background detection: 14.3% of counts in low region
  Recommendation: two_stage
  Using two-stage fitting with background subtraction...
    Fitting background Gaussian in range [0, 0.07] mV·µs
User defined signal region: 

/Users/pabloherrero/sabat/RaTagging/RaTag/core/fitting.py:303: RuntimeWarning: invalid value encountered in power
  tail = A_tail / (denom_safe)**m


  → Saved: /Volumes/KINGSTON/RaTag_data/processed/RUN26_Th228_EL2700/plots/s2_areas/FieldScan_Gate1000_Anode3700_s2_histogram.png
    📊 Saved histogram plot
Processing set FieldScan_Gate1200_Anode3900...
  Background detection: 19.3% of counts in low region
  Recommendation: two_stage
  Using two-stage fitting with background subtraction...
    Fitting background Gaussian in range [0, 0.07] mV·µs
User defined signal region: [0.05, 5.0]
    ✓ Fit (two_stage): μ=2.081 ± 0.000 mV·µs
    💾 Saved explicit fit result to FieldScan_Gate1200_Anode3900_s2_areas_hist_fit.json


/Users/pabloherrero/sabat/RaTagging/RaTag/core/fitting.py:303: RuntimeWarning: invalid value encountered in power
  tail = A_tail / (denom_safe)**m


  → Saved: /Volumes/KINGSTON/RaTag_data/processed/RUN26_Th228_EL2700/plots/s2_areas/FieldScan_Gate1200_Anode3900_s2_histogram.png
    📊 Saved histogram plot
